In [1]:
import pandas as pd
import numpy as np
from scipy.stats import poisson
from itertools import combinations #automatically creates all unique team pairs
#for eg: teams = ['Brazil', 'Germany', 'France', 'Japan']
#combinations(teams, 2) produces: Brazil vs Germany, Brazil Vs France, Brazil vs Japan, Germany vs France, Germany vs Japan, France vs Japan

In [8]:
team_stats = pd.read_csv('../data/team_stats.csv')

def predict_match(home_team, away_team, team_stats, max_goals = 6):
    """
    Predicting the scoreline between two teams.
    Returns the expected goals and win probabilities
    """
    home = team_stats[team_stats['team']==home_team].iloc[0]
    away = team_stats[team_stats['team']==away_team].iloc[0]

    #calculating expected goals
    #Home expected goals
    # Dividing by 0.9 which is < 1 to slightly increase home scoring
    #because home teams usually performs better (home advantage assumption)
    home_expected_goals = (
        home['home_goals_scored'] * away['away_goals_conceded'] / 0.9
    )

    # /1.1 to slightly reduce scoring of away team
    away_expected_goals = (
        away['away_goals_scored'] * home['home_goals_conceded'] / 1.1
    )

    #Clipping to reasonable range of expected goals
    #np.clip(value, min, max)
    home_expected_goals = np.clip(home_expected_goals, 0.3, 4.0)
    away_expected_goals = np.clip(away_expected_goals, 0.3, 4.0)

    # Using poisson to build prob matrix
    home_prob = [poisson.pmf(i, home_expected_goals) for i in range(max_goals+1)]
    away_prob = [poisson.pmf(i, away_expected_goals) for i in range(max_goals+1)]

    #Building the score prob matrix
    #np.outer(a, b, out=None(optional)) computes the outer product of two vectors
    #a:[array_like], first input vector, input is flattened if not in 1D
    #b:[array_like], second input vector, input is flattened if not in 1D
    #out:[ndarray], it is a location where the result is stored
    score_matrix = np.outer(home_prob, away_prob)

    #Win probabilities
    home_win_prob = np.tril(score_matrix, -1).sum() #returns lower triangle of matrix
    draw_prob = np.trace(score_matrix) # diagonal elements
    away_win_prob = np.triu(score_matrix, 1).sum() #returns the upper triangle of matrix

    max_idx = np.unravel_index(score_matrix.argmax(), score_matrix.shape)
    predicted_home_goals = max_idx[0]
    predicted_away_goals = max_idx[1]


    if home_win_prob > away_win_prob and home_win_prob > draw_prob:
        winner = "home"
    elif away_win_prob > home_win_prob and away_win_prob > draw_prob:
        winner = "away"
    else:
        winner = "draw"

    return{
        'home_team':home_team,
        'away_team':away_team,
        'predicted_home_goals':predicted_home_goals,
        'predicted_away_goals':predicted_away_goals,
        'home_expected_goals':round(home_expected_goals, 2),
        'away_expected_goals':round(away_expected_goals, 2),
        'home_win_prob':round(home_win_prob * 100, 1),
        'draw_prob':round(draw_prob * 100, 1),
        'away_win_prob':round(away_win_prob * 100, 1),
        'predicted_winner':winner
    }

#Testing
result = predict_match("Brazil", "Germany", team_stats)
print(result)

AVG_CORNERS = 10.2
AVG_YELLOW = 3.1
AVG_RED = 0.2

def predict_extras(home_team, away_team, team_stats):
    """
    Predicting corners, yellow cards, red cards for a match.
    High-pressure matches -> more cards
    """
    home= team_stats[team_stats['team']==home_team].iloc[0]
    away= team_stats[team_stats['team']==away_team].iloc[0]

    #using rank_factor as assumption strong teams or top teams play cleaner
    rank_factor = (home['fifa_rank'] + away['fifa_rank']) / 200

    corners = round(AVG_CORNERS + (rank_factor * 2), 0)
    yellow_cards = round(AVG_YELLOW + (rank_factor * 0.5), 0)
    red_cards = 0

    return{
        'corners':int(corners),
        'yellow_cards':int(yellow_cards),
        'red_cards':int(red_cards)
    }


{'home_team': 'Brazil', 'away_team': 'Germany', 'predicted_home_goals': np.int64(3), 'predicted_away_goals': np.int64(1), 'home_expected_goals': np.float64(4.0), 'away_expected_goals': np.float64(1.2), 'home_win_prob': np.float64(73.9), 'draw_prob': np.float64(8.7), 'away_win_prob': np.float64(6.3), 'predicted_winner': 'home'}


In [9]:
groups = {
    'A': ['Mexico', 'South Africa', 'South Korea', 'Czechia'],
    'B': ['Canada', 'Bosnia and Herzegovina', 'Qatar', 'Switzerland'],
    'C': ['Brazil', 'Morocco', 'Haiti', 'Scotland'],
    'D': ['USA', 'Paraguay', 'Australia', 'Türkiye'],
    'E': ['Germany', 'Curaçao', "Côte d'Ivoire"],
    'F': ['Netherlands','Japan', 'Sweden', 'Tunisia'],
    'G': ['Belgium', 'Egypt', 'Iran', 'New Zealand'],
    'H': ['Spain','Cabo Verde', 'Saudi Arabia', 'Uruguay'],
    'I': ['France', 'Senegal', 'Iraq', 'Norway'],
    'J': ['Argentina', 'Algeria', 'Austria', 'Jordan'],
    'K': ['Portugal', 'Democratic Republic of Congo', 'Uzbekistan', 'Colombia'],
    'L': ['England', 'Croatia', 'Ghana', 'Panama'],
}

print("Groups defined: ")
for group, teams in groups.items():
    print(f" Group {group}:{', '.join(teams)}")

Groups defined: 
 Group A:Mexico, South Africa, South Korea, Czechia
 Group B:Canada, Bosnia and Herzegovina, Qatar, Switzerland
 Group C:Brazil, Morocco, Haiti, Scotland
 Group D:USA, Paraguay, Australia, Türkiye
 Group E:Germany, Curaçao, Côte d'Ivoire
 Group F:Netherlands, Japan, Sweden, Tunisia
 Group G:Belgium, Egypt, Iran, New Zealand
 Group H:Spain, Cabo Verde, Saudi Arabia, Uruguay
 Group I:France, Senegal, Iraq, Norway
 Group J:Argentina, Algeria, Austria, Jordan
 Group K:Portugal, Democratic Republic of Congo, Uzbekistan, Colombia
 Group L:England, Croatia, Ghana, Panama


In [10]:
def simulate_group(group_name, teams, team_stats):
    """
    Play every team against every other team in the group.
    Return all match results.
    """
    matches = []
    match_id = 1
    
    # Every pair of teams plays once
    for home_team, away_team in combinations(teams, 2):
        pred = predict_match(home_team, away_team, team_stats)
        extras = predict_extras(home_team, away_team, team_stats)
        
        matches.append({
            'group': group_name,
            'home_team': home_team,
            'away_team': away_team,
            'home_goals': pred['predicted_home_goals'],
            'away_goals': pred['predicted_away_goals'],
            'winner': pred['predicted_winner'],
            'corners': extras['corners'],
            'yellow_cards': extras['yellow_cards'],
            'red_cards': extras['red_cards'],
        })
    
    return matches

# Test with Group A
group_a_matches = simulate_group('A', groups['A'], team_stats)
print("Group A matches:")
for m in group_a_matches:
    print(f"  {m['home_team']} {m['home_goals']}-{m['away_goals']} {m['away_team']}  → {m['winner']} wins")

Group A matches:
  Mexico 2-1 South Africa  → home wins
  Mexico 1-1 South Korea  → home wins
  Mexico 0-0 Czechia  → draw wins
  South Africa 1-0 South Korea  → home wins
  South Africa 0-0 Czechia  → draw wins
  South Korea 0-0 Czechia  → draw wins
